In [77]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound, TranscriptsDisabled, VideoUnavailable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os


Step 1a - indexing (Documents Ingestion) - YouTube Transcript

In [78]:
video_id = "0Rwb4Xmlcwc"  # Replace with the actual YouTube video

if video_id == "YOUR_VIDEO_ID":
    raise ValueError("Replace video_id with the YouTube video ID before running this cell.")

try:
    # Fetch the preferred English transcript.
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=("en",))

    # Flatten the transcript snippets to plain text.
    transcript_text = " ".join(snippet.text for snippet in transcript)
    print("Transcript fetched successfully.", transcript_text[:500])
except (NoTranscriptFound, TranscriptsDisabled):
    raise RuntimeError(f"No transcript is available for video {video_id!r}.")
except VideoUnavailable:
    raise RuntimeError(f"Video {video_id!r} is unavailable.")


Transcript fetched successfully. backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different 


Step 1b - indexing (Documents Ingestion) - YouTube Transcript

In [79]:
splitters = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitters.create_documents([transcript_text])

In [80]:
len(chunks)  # Check how many chunks were created

41

In [81]:
chunks[0].page_content[:500]  # Display the first 500 characters of the first chunk

'backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different '

Step 1c & 1d - Indexing(Embedding creation and Vector Store Creation) - YouTube Transcript

In [82]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=30,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

vector_store = FAISS.from_documents(chunks, embedding_model)


In [83]:
vector_store.index_to_docstore_id

{0: 'c09d4e68-cab3-4045-bda6-b3c80dfdbd30',
 1: '63117f77-ed53-4014-acbe-ad17b0f43cc9',
 2: '652929db-9941-4a9e-a391-e7fa30e7582d',
 3: 'b6e8639d-2b24-46e1-82c7-0fc8a06c8975',
 4: '93c00795-ddeb-4bc6-a46b-6d525830d0aa',
 5: '778f2131-d347-4d32-af39-3becb28792d8',
 6: '39ebeaeb-a3eb-49ee-b9b8-73c16976672b',
 7: 'dba1a8cd-4d09-4c54-b821-abd8ca679e4e',
 8: 'e5db1e98-8765-4a54-b541-5997848fe76b',
 9: 'c2dd063b-d1b1-4cca-bfa5-7ac22119cca2',
 10: '7334a658-19c9-460f-9cfc-051740fee70a',
 11: '7e586c75-d661-4de8-a4e1-98dc67f5111c',
 12: '80951775-3c1f-4232-b203-d326923338cf',
 13: '0ce5ff47-5ee1-4ac0-9976-36d0d6436dd2',
 14: '2e75c975-0f7e-43d9-8388-b64d4999d7c5',
 15: '1a245cfd-6be3-4f29-b0f7-15d2f6acb21f',
 16: 'efd1579f-9dcc-41b8-ad82-dd0b3923c4a4',
 17: '9e5341da-df9e-4bca-adc7-52cb9807d105',
 18: 'ff70a206-131c-4509-aad0-d37ad00abd71',
 19: 'd6f21259-e7f5-4559-ad38-2dabc6eaeb5b',
 20: '7a3a3901-4d9c-47da-a1d4-7a7c7ae74680',
 21: '6b191ed4-0d74-41c2-98f4-d75bcbe58c02',
 22: 'fbdf3af3-02ce-

In [84]:
vector_store.get_by_ids(["77b82958-c5f2-449f-8142-ecac2ffa9de5"]) # Retrieve the first three documents by their IDs

[]

Step 2 - Retrieval and Question Answering - YouTube Transcript

In [85]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 4})

In [86]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7080907256d0>, search_type='mmr', search_kwargs={'k': 4})

In [87]:
retriever.invoke("What is backend from first principles?")  # Ask a question to the retriever

[Document(id='c09d4e68-cab3-4045-bda6-b3c80dfdbd30', metadata={}, page_content='backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different concepts come together and this is the reason it takes people years to get their head around a lot of these Concepts and principles because people primarily start with a limited scope of training uh whether it is from a college or a boot camp or a simple cop course and they eventually build onp top of that with trial and error and with help of other developers over time now I am a backend engineer and I have faced th

Step 3 - Augmented Generation - YouTube Transcript

In [88]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant that answers questions based on the context provided.
    If the answer is not contained within the context, say "I don't know."
    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [89]:
question = "what topics are covered in the video?"
retriever_docs = retriever.invoke(question)

In [90]:
for doc in retriever_docs:  # Display the retrieved documents
    print(f"Document: {doc.page_content[:100]}...")

Document: we'll see how ORS work whether to use an orm what are the tradeoffs and we'll look at what are datab...
Document: side caching and server side caching the different caching strategies for example cash aside right t...
Document: and efficient ensuring middleware is applied in the correct order or how how middleware order can af...
Document: switch to a different language let's say you were working with rubyan rails for years and one day yo...


In [91]:
context = "\n\n".join([doc.page_content for doc in retriever_docs])

In [92]:
print("Context:\n", context)  # Display the first 500 characters of the context

Context:
 we'll see how ORS work whether to use an orm what are the tradeoffs and we'll look at what are database migrations that we'll move on to business logic layer which is also called BL what is the role of it what are the different layers of a request cycle for example we have validation layer we have routing we have middlewares and we have handl and controls which all of them fall under presentation layer because they they deal with users data whether it is accepting users data or sending a user data so those are part of a presentation layer after that we have business logic layer which is the middle one which deals with our Core Business logic and after that we have the data access layer which deals with databases perform squaring or inserts or deletions and business logic layer uses the data access layer behind the scenes we'll look at different design principles like separation of concerns single responsibility open close dependency inversion what are the components of a busi

In [93]:
final_prompt = prompt.invoke({"context": context, "question": question})

Step 5 - Generation of Answer - YouTube Transcript

In [94]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [95]:
answer = model.invoke(final_prompt)

In [96]:
print(answer.content)

The video covers a range of topics related to backend systems, including:

1. **ORMs (Object-Relational Mappers)** - Discussion on how ORMs work, their trade-offs, and database migrations.
2. **Business Logic Layer (BL)** - The role of the BL, different layers of a request cycle, and components like validation, routing, middlewares, and handlers that fall under the presentation layer.
3. **Design Principles** - Concepts like separation of concerns, single responsibility, open/closed principle, and dependency inversion.
4. **Caching** - Different strategies for caching including side caching, server-side caching, cache eviction strategies (LRU, LFU, TTL, FIFO), and the need for cache invalidation methods.
5. **Request Context** - Metadata passed through middlewares, controllers, and services, life cycle of a request, maintaining state, and components of a request context like HTTP method, URL, headers, query parameters, body, and session/user information.
6. **Migration from Ruby on Rai

In [97]:
from langchain_classic.retrievers import ContextualCompressionRetriever  # noqa: F401
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


In [98]:
compressor = LLMChainExtractor.from_llm(llm=model)


In [99]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)


In [100]:
compressed_docs = compression_retriever.invoke(question)
compressed_context = "\n\n".join(doc.page_content for doc in compressed_docs)
compressed_prompt = prompt.invoke(
    {"context": compressed_context, "question": question}
)
answer = model.invoke(compressed_prompt)

In [101]:
print(answer.content)

The video covers the following topics:

1. ORS (Object-Relational Mapping) and its trade-offs.
2. Database migrations.
3. Business Logic Layer (BL) and its role in the request cycle.
4. Different layers of the request cycle: validation layer, routing, middleware, and handlers/controllers (presentation layer).
5. Data Access Layer and its interaction with the Business Logic Layer.
6. Design principles: separation of concerns, single responsibility, open/close principle, and dependency inversion.
7. Caching strategies: side caching, server-side caching, cache-aside, right-through, right-behind, and right-back.
8. Cache eviction strategies: LRU (Least Recently Used), LFU (Least Frequently Used), TTL (Time To Live), and FIFO (First In, First Out).
9. Cache invalidation methods: manual, time-to-live, and event-based invalidation.
10. Levels of caching: level one (in-memory) and level two (network distributed), and hierarchical caching.
11. Caching for web applications, including caching sta